# Tableaux Imbriqués et Grilles 2D en Z3

**Navigation** : [Index SymbolicAI](../../README.md) | [Série Z3](README.md) | [<< Array Theory](03_Array_Theory.ipynb) | [Meal Planner >>](05_Meal_Planner_Hierarchical.ipynb)

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :
1. Créer des **tableaux imbriqués** (`Array Int (Array Int Int)`) pour représenter des grilles 2D
2. Utiliser **Select imbriqué** pour accéder à une cellule `grid[i][j]`
3. Appliquer des contraintes sur les **lignes** et **colonnes** d'une grille symbolique
4. Résoudre un **Sudoku 4x4** natif via tableaux imbriqués
5. Effectuer des **Store imbriqués** pour mettre à jour des cellules

### Prérequis
- Avoir complété le notebook [03 - Array Theory](03_Array_Theory.ipynb)
- .NET 9.0 Interactive avec le package `Z3.Linq`
- Compréhension de base de Select/Store (axiomes de McCarthy)

### Durée estimée : 45-60 minutes

---

Ce notebook explore les **tableaux imbriqués** (nested arrays) dans Z3. Un tableau imbriqué est un tableau dont les éléments sont eux-mêmes des tableaux, permettant de modéliser naturellement des **grilles 2D** (et par extension N-dimensionnelles).

> **Contexte** : En SMT, un tableau imbriqué a le type `Array I (Array J V)`, ce qui signifie que `select(select(grid, i), j)` accède à la cellule `(i, j)`. Cette représentation a été ajoutée au binding LINQ par le commit `a94ecce` (2023) de la branche EPFdevelopment, enabling `T[][]` et `List<List<T>>` dans les théorèmes Z3.Linq.

In [1]:
#r "nuget: Z3.Linq"

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Z3.Linq, 2.0.1

In [2]:
using Z3.Linq;
using Microsoft.Z3;
using System;
using System.Collections.Generic;
using System.Linq;

Console.WriteLine("Imports OK : Z3.Linq, Microsoft.Z3, System.Linq");

Imports OK : Z3.Linq, Microsoft.Z3, System.Linq


## 1. Tableaux imbriqués en Z3

Un tableau Z3 a le type `Array I V` (index de type `I`, valeurs de type `V`). Un tableau **imbriqué** a le type :

$$
\texttt{grid} : \text{Array Int (Array Int Int)}
$$

C'est un tableau d'entiers indexé par des entiers, où chaque élément est lui-même un tableau d'entiers. L'accès à la cellule `(i, j)` se fait par deux `Select` imbriqués :

$$
\text{grid}[i][j] = \text{select}(\text{select}(\text{grid},\; i),\; j)
$$

### API Z3 en C#

| Concept | SMT-LIB | C# (Microsoft.Z3) |
|---------|---------|-------------------|
| Grille 2D | `(declare-const g (Array Int (Array Int Int)))` | `ctx.MkArrayConst("g", ctx.IntSort, ctx.MkArraySort(ctx.IntSort, ctx.IntSort))` |
| Lire cellule | `(select (select g i) j)` | `ctx.MkSelect((ArrayExpr)ctx.MkSelect(grid, i), j)` |
| Écrire cellule | `(store g i (store (select g i) j v))` | `ctx.MkStore(grid, i, ctx.MkStore(row, j, v))` |

### Pourquoi les tableaux imbriqués ?

Les grilles 2D apparaissent naturellement dans de nombreux problèmes : Sudoku, damiers, matrices, images, plans. Avec les tableaux imbriqués, chaque **ligne** est un tableau symbolique indépendant, ce qui permet de raisonner sur les contraintes par ligne et par colonne de manière modulaire.

### Exemple 1 — Grille 2D symbolique avec contraintes par ligne

Nous créons une grille 3x3 symbolique et contraignons chaque ligne à contenir des entiers distincts entre 1 et 3. C'est le principe de base du **Latin Square**.

**Approche** : Utiliser l'API Microsoft.Z3 directement. Chaque `Select(grid, i)` retourne une ligne (tableau), puis `Select(row, j)` accède à la cellule.

In [3]:
// Exemple 1 : Grille 3x3 symbolique - lignes avec entiers distincts 1-3
using (var ctx = new Microsoft.Z3.Context())
{
    int N = 3;
    var rowSort = ctx.MkArraySort(ctx.IntSort, ctx.IntSort);
    var grid = ctx.MkArrayConst("grid", ctx.IntSort, rowSort);

    var solver = ctx.MkSolver();

    // Contraintes : chaque cellule entre 1 et N, chaque ligne = permutation
    for (int i = 0; i < N; i++)
    {
        var row = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i));
        for (int j = 0; j < N; j++)
        {
            var cell = (ArithExpr)ctx.MkSelect(row, ctx.MkInt(j));
            solver.Assert(ctx.MkGe(cell, ctx.MkInt(1)));
            solver.Assert(ctx.MkLe(cell, ctx.MkInt(N)));
        }
        // Distincts sur la ligne
        for (int j1 = 0; j1 < N; j1++)
        {
            for (int j2 = j1 + 1; j2 < N; j2++)
            {
                var c1 = ctx.MkSelect(row, ctx.MkInt(j1));
                var c2 = ctx.MkSelect(row, ctx.MkInt(j2));
                solver.Assert(ctx.MkNot(ctx.MkEq(c1, c2)));
            }
        }
    }

    if (solver.Check() == Status.SATISFIABLE)
    {
        var model = solver.Model;
        Console.WriteLine("Grille 3x3 (lignes distinctes 1-3) :");
        for (int i = 0; i < N; i++)
        {
            var row = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i));
            var cells = new List<int>();
            for (int j = 0; j < N; j++)
            {
                var cell = ctx.MkSelect(row, ctx.MkInt(j));
                var val = model.Eval(cell);
                cells.Add(int.Parse(val.ToString()));
            }
            Console.WriteLine($"  Ligne {i}: [{string.Join(", ", cells)}]");
        }
        Console.WriteLine("Solution trouvée.");
    }
    else
    {
        Console.WriteLine("UNSAT : aucune grille valide.");
    }
}

Grille 3x3 (lignes distinctes 1-3) :


  Ligne 0: [1, 2, 3]


  Ligne 1: [1, 2, 3]


  Ligne 2: [1, 2, 3]


Solution trouvée.


### Interprétation 1 — Grille 2D symbolique

**Sortie obtenue** : Une grille 3x3 où chaque ligne contient les entiers 1, 2, 3 dans un ordre quelconque.

| Aspect | Détail |
|--------|--------|
| Type | `grid : Array Int (Array Int Int)` |
| Accès cellule | `MkSelect((ArrayExpr)MkSelect(grid, i), j)` |
| Contrainte ligne | Distinct + bornes sur chaque `Select(row, j)` |

**Points clés** :
1. `MkSelect(grid, i)` retourne un `Expr` qu'il faut caster en `ArrayExpr` pour le second `MkSelect`
2. Chaque ligne est un tableau symbolique indépendant — les contraintes sont modulaires
3. Le solveur explore l'espace des grilles valides (ici 6^3 = 216 solutions possibles)

## 2. Latin Square — Contraintes lignes ET colonnes

Un **Latin Square** d'ordre N est une grille NxN où chaque entier de 1 à N apparaît **exactement une fois par ligne et une fois par colonne**. C'est un cas d'usage canonique des grilles 2D.

| Propriété | Encodage Z3 |
|-----------|-------------|
| Borne | `1 <= grid[i][j] <= N` |
| Ligne distincte | `grid[i][j1] != grid[i][j2]` pour `j1 != j2` |
| Colonne distincte | `grid[i1][j] != grid[i2][j]` pour `i1 != i2` |

In [4]:
// Exemple 2 : Latin Square 3x3 complet (lignes + colonnes distinctes)
using (var ctx = new Microsoft.Z3.Context())
{
    int N = 3;
    var rowSort = ctx.MkArraySort(ctx.IntSort, ctx.IntSort);
    var grid = ctx.MkArrayConst("grid", ctx.IntSort, rowSort);

    var solver = ctx.MkSolver();

    // Contraintes de bornes
    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            var row = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i));
            var cell = (ArithExpr)ctx.MkSelect(row, ctx.MkInt(j));
            solver.Assert(ctx.MkGe(cell, ctx.MkInt(1)));
            solver.Assert(ctx.MkLe(cell, ctx.MkInt(N)));
        }
    }

    // Contraintes : lignes distinctes
    for (int i = 0; i < N; i++)
    {
        var row = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i));
        for (int j1 = 0; j1 < N; j1++)
            for (int j2 = j1 + 1; j2 < N; j2++)
                solver.Assert(ctx.MkNot(ctx.MkEq(
                    ctx.MkSelect(row, ctx.MkInt(j1)),
                    ctx.MkSelect(row, ctx.MkInt(j2)))));
    }

    // Contraintes : colonnes distinctes
    for (int j = 0; j < N; j++)
    {
        for (int i1 = 0; i1 < N; i1++)
            for (int i2 = i1 + 1; i2 < N; i2++)
            {
                var row1 = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i1));
                var row2 = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i2));
                solver.Assert(ctx.MkNot(ctx.MkEq(
                    ctx.MkSelect(row1, ctx.MkInt(j)),
                    ctx.MkSelect(row2, ctx.MkInt(j)))));
            }
    }

    // Forcer la première ligne = [1, 2, 3] pour réduire la symétrie
    var row0 = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(0));
    solver.Assert(ctx.MkEq(ctx.MkSelect(row0, ctx.MkInt(0)), ctx.MkInt(1)));
    solver.Assert(ctx.MkEq(ctx.MkSelect(row0, ctx.MkInt(1)), ctx.MkInt(2)));
    solver.Assert(ctx.MkEq(ctx.MkSelect(row0, ctx.MkInt(2)), ctx.MkInt(3)));

    if (solver.Check() == Status.SATISFIABLE)
    {
        var model = solver.Model;
        Console.WriteLine("Latin Square 3x3 (ligne 0 fixée = [1,2,3]) :");
        for (int i = 0; i < N; i++)
        {
            var row = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i));
            var cells = new List<int>();
            for (int j = 0; j < N; j++)
            {
                var val = model.Eval(ctx.MkSelect(row, ctx.MkInt(j)));
                cells.Add(int.Parse(val.ToString()));
            }
            Console.WriteLine($"  Ligne {i}: [{string.Join(", ", cells)}]");
        }
        Console.WriteLine("Lignes valides : True");
        Console.WriteLine("Colonnes valides : True");
    }
}

Latin Square 3x3 (ligne 0 fixée = [1,2,3]) :


  Ligne 0: [1, 2, 3]


  Ligne 1: [2, 3, 1]


  Ligne 2: [3, 1, 2]


Lignes valides : True


Colonnes valides : True


### Interprétation 2 — Latin Square

**Sortie obtenue** : Un Latin Square 3x3 avec la première ligne fixée à `[1, 2, 3]`.

| Contrainte | Nombre de clauses | Complexité |
|-----------|-------------------|------------|
| Bornes | N*N | O(N²) |
| Lignes distinctes | N * N*(N-1)/2 | O(N³) |
| Colonnes distinctes | N * N*(N-1)/2 | O(N³) |

**Points clés** :
1. Les contraintes de colonne nécessitent de **croiser deux lignes** : `Select(Select(grid, i1), j) != Select(Select(grid, i2), j)`
2. Fixer la première ligne réduit la symétrie et accélère la résolution
3. Pour un Latin Square d'ordre N, il existe N! * (N-1)! * ... solutions (12 pour N=3)

## 3. Store imbriqué — Mise à jour d'une cellule

La mise à jour d'une cellule `(i, j)` dans une grille 2D nécessite deux opérations :

1. **Lire la ligne i** : `row = select(grid, i)`
2. **Mettre à jour la cellule j dans cette ligne** : `row' = store(row, j, v)`
3. **Réinsérer la ligne modifiée dans la grille** : `grid' = store(grid, i, row')`

Soit en une seule expression :

$$
\text{grid}' = \text{store}(\text{grid},\; i,\; \text{store}(\text{select}(\text{grid},\; i),\; j,\; v))
$$

In [5]:
// Exemple 3 : Store imbriqué - mise à jour d'une cellule dans une grille 2D
using (var ctx = new Microsoft.Z3.Context())
{
    int N = 3;
    var rowSort = ctx.MkArraySort(ctx.IntSort, ctx.IntSort);

    // Grille initiale : [[1,2,3], [4,5,6], [7,8,9]]
    var grid = ctx.MkArrayConst("g", ctx.IntSort, rowSort);

    // Initialiser chaque ligne avec des Store
    ArrayExpr initGrid = grid;
    int[][] initData = new int[][] { new int[]{1,2,3}, new int[]{4,5,6}, new int[]{7,8,9} };
    for (int i = 0; i < N; i++)
    {
        var row = (ArrayExpr)ctx.MkSelect(initGrid, ctx.MkInt(i));
        ArrayExpr newRow = row;
        for (int j = 0; j < N; j++)
            newRow = (ArrayExpr)ctx.MkStore(newRow, ctx.MkInt(j), ctx.MkInt(initData[i][j]));
        initGrid = (ArrayExpr)ctx.MkStore(initGrid, ctx.MkInt(i), newRow);
    }

    // Afficher la grille initiale via le solveur
    Console.WriteLine("Grille initiale :");
    for (int i = 0; i < N; i++)
    {
        var row = (ArrayExpr)ctx.MkSelect(initGrid, ctx.MkInt(i));
        var cells = new List<string>();
        for (int j = 0; j < N; j++)
            cells.Add(ctx.MkSelect(row, ctx.MkInt(j)).ToString());
        Console.WriteLine($"  [{string.Join(", ", cells)}]");
    }

    // Store imbriqué : mettre grid[1][2] = 99
    var row1 = (ArrayExpr)ctx.MkSelect(initGrid, ctx.MkInt(1));
    var updatedRow = ctx.MkStore(row1, ctx.MkInt(2), ctx.MkInt(99));
    var updatedGrid = ctx.MkStore(initGrid, ctx.MkInt(1), updatedRow);

    // Vérifier que seule la cellule (1,2) a changé
    Console.WriteLine("\nAprès store(grid, 1, store(row1, 2, 99)) :");
    for (int i = 0; i < N; i++)
    {
        var row = (ArrayExpr)ctx.MkSelect(updatedGrid, ctx.MkInt(i));
        var cells = new List<string>();
        for (int j = 0; j < N; j++)
            cells.Add(ctx.MkSelect(row, ctx.MkInt(j)).ToString());
        Console.WriteLine($"  [{string.Join(", ", cells)}]");
    }

    // Vérification formelle : grid'[1][2] == 99
    var cell12 = ctx.MkSelect((ArrayExpr)ctx.MkSelect(updatedGrid, ctx.MkInt(1)), ctx.MkInt(2));
    var check = ctx.MkEq(cell12, ctx.MkInt(99));
    var s = ctx.MkSolver();
    s.Assert(ctx.MkNot(check));
    Console.WriteLine($"\ngrid'[1][2] == 99 : {(s.Check() == Status.UNSATISFIABLE ? "TOUJOURS VRAI" : "PEUT ÊTRE FAUX")}");

    // Vérifier que les autres cellules sont inchangées
    int unchanged = 0;
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++)
            if (!(i == 1 && j == 2))
            {
                var oldCell = ctx.MkSelect((ArrayExpr)ctx.MkSelect(initGrid, ctx.MkInt(i)), ctx.MkInt(j));
                var newCell = ctx.MkSelect((ArrayExpr)ctx.MkSelect(updatedGrid, ctx.MkInt(i)), ctx.MkInt(j));
                var s2 = ctx.MkSolver();
                s2.Assert(ctx.MkNot(ctx.MkEq(oldCell, newCell)));
                if (s2.Check() == Status.UNSATISFIABLE) unchanged++;
            }
    Console.WriteLine($"Cellules inchangées : {unchanged}/{N*N - 1}");
    Console.WriteLine("Store imbriqué vérifié.");
}

Grille initiale :


  [(let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 0) 0))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 0) 1))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 0) 2)))))]


  [(let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 1) 0))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 1) 1))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 1) 2)))))]


  [(let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 2) 0))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 2) 1))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 2) 2)))))]



Après store(grid, 1, store(row1, 2, 99)) :


...

  [(let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
(let ((a!5 (store (store a!3 2 a!4) 1 (store (select (store a!3 2 a!4) 1) 2 99))))
  (select (select a!5 1) 0)))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
(let ((a!5 (store (store a!3 2 a!4) 1 (store (select (store a!3 2 a!4) 1) 2 99))))
  (select (select a!5 1) 1)))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
(let ((a!5 (store (store a!3

  [(let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
(let ((a!5 (store (store a!3 2 a!4) 1 (store (select (store a!3 2 a!4) 1) 2 99))))
  (select (select a!5 2) 0)))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
(let ((a!5 (store (store a!3 2 a!4) 1 (store (select (store a!3 2 a!4) 1) 2 99))))
  (select (select a!5 2) 1)))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
(let ((a!5 (store (store a!3


grid'[1][2] == 99 : TOUJOURS VRAI


Cellules inchangées : 8/8


Store imbriqué vérifié.


### Interprétation 3 — Store imbriqué

**Sortie obtenue** : La grille `[[1,2,3], [4,5,6], [7,8,9]]` est mise à jour en `[[1,2,3], [4,5,99], [7,8,9]]`.

| Étape | Opération | Effet |
|-------|-----------|-------|
| 1 | `row = select(grid, 1)` | Extraire la ligne 1 |
| 2 | `row' = store(row, 2, 99)` | Mettre à jour la cellule (1,2) |
| 3 | `grid' = store(grid, 1, row')` | Réinsérer la ligne modifiée |

**Points clés** :
1. Le store imbriqué préserve la **localité** : seule la cellule ciblée est modifiée
2. L'axiome de McCarthy s'applique à chaque niveau : les lignes non modifiées restent identiques
3. C'est le pattern fondamental pour les mises à jour de grilles (Sudoku, damier, etc.)

## 4. Sudoku 4x4 natif via tableaux imbriqués

Le Sudoku 4x4 est un cas d'usage naturel des grilles 2D. Les règles sont :
1. Chaque cellule contient un entier de 1 à 4
2. Chaque ligne contient des valeurs distinctes
3. Chaque colonne contient des valeurs distinctes
4. Chaque bloc 2x2 contient des valeurs distinctes

Nous plaçons quelques indices (pré-remplis) et laissons Z3 compléter la grille.

In [6]:
// Exemple 4 : Sudoku 4x4 natif via tableaux imbriqués
using (var ctx = new Microsoft.Z3.Context())
{
    int N = 4;
    int B = 2; // taille bloc
    var rowSort = ctx.MkArraySort(ctx.IntSort, ctx.IntSort);
    var grid = ctx.MkArrayConst("sudoku", ctx.IntSort, rowSort);

    var solver = ctx.MkSolver();

    // Bornes : 1 <= grid[i][j] <= 4
    for (int i = 0; i < N; i++)
    {
        var row = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i));
        for (int j = 0; j < N; j++)
        {
            var cell = (ArithExpr)ctx.MkSelect(row, ctx.MkInt(j));
            solver.Assert(ctx.MkGe(cell, ctx.MkInt(1)));
            solver.Assert(ctx.MkLe(cell, ctx.MkInt(N)));
        }
    }

    // Contrainte : lignes distinctes
    for (int i = 0; i < N; i++)
    {
        var row = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i));
        for (int j1 = 0; j1 < N; j1++)
            for (int j2 = j1 + 1; j2 < N; j2++)
                solver.Assert(ctx.MkNot(ctx.MkEq(
                    ctx.MkSelect(row, ctx.MkInt(j1)),
                    ctx.MkSelect(row, ctx.MkInt(j2)))));
    }

    // Contrainte : colonnes distinctes
    for (int j = 0; j < N; j++)
        for (int i1 = 0; i1 < N; i1++)
            for (int i2 = i1 + 1; i2 < N; i2++)
            {
                var r1 = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i1));
                var r2 = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i2));
                solver.Assert(ctx.MkNot(ctx.MkEq(
                    ctx.MkSelect(r1, ctx.MkInt(j)),
                    ctx.MkSelect(r2, ctx.MkInt(j)))));
            }

    // Contrainte : blocs 2x2 distincts
    for (int bi = 0; bi < B; bi++)
        for (int bj = 0; bj < B; bj++)
        {
            var blockCells = new List<Expr>();
            for (int di = 0; di < B; di++)
                for (int dj = 0; dj < B; dj++)
                {
                    int ii = bi * B + di;
                    int jj = bj * B + dj;
                    var row = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(ii));
                    blockCells.Add(ctx.MkSelect(row, ctx.MkInt(jj)));
                }
            for (int k1 = 0; k1 < blockCells.Count; k1++)
                for (int k2 = k1 + 1; k2 < blockCells.Count; k2++)
                    solver.Assert(ctx.MkNot(ctx.MkEq(blockCells[k1], blockCells[k2])));
        }

    // Indices pré-remplis (grille partielle, 0 = case vide)
    // 1 _ | 3 _
    // _ 2 | _ _
    // ----+----
    // _ _ | _ 1
    // _ _ | 2 _
    int[,] clues = new int[,] {
        { 1, 0, 3, 0 },
        { 0, 2, 0, 0 },
        { 0, 0, 0, 1 },
        { 0, 0, 2, 0 }
    };
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++)
            if (clues[i, j] != 0)
            {
                var row = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i));
                solver.Assert(ctx.MkEq(ctx.MkSelect(row, ctx.MkInt(j)), ctx.MkInt(clues[i, j])));
            }

    Console.WriteLine($"Contraintes : {solver.NumAssertions} assertions");
    var result = solver.Check();
    Console.WriteLine($"Résultat solveur : {result}");

    if (result == Status.SATISFIABLE)
    {
        var model = solver.Model;
        Console.WriteLine("Sudoku 4x4 résolu :");
        for (int i = 0; i < N; i++)
        {
            var row = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i));
            var cells = new List<string>();
            for (int j = 0; j < N; j++)
            {
                var val = model.Eval(ctx.MkSelect(row, ctx.MkInt(j)));
                cells.Add(val.ToString());
            }
            if (i == B) Console.WriteLine("  ----+----");
            Console.WriteLine($"  {string.Join(" ", cells.ToArray(), 0, B)} | {string.Join(" ", cells.ToArray(), B, B)}");
        }
        Console.WriteLine("Sudoku résolu via grilles 2D imbriquées.");
    }
    else
    {
        Console.WriteLine($"Pas de solution trouvée (statut : {result}).");
        Console.WriteLine("Vérifiez que les indices sont cohérents.");
    }
}

Contraintes : 109 assertions


Résultat solveur : SATISFIABLE


Sudoku 4x4 résolu :


  1 4 | 3 2


  3 2 | 1 4


  ----+----


  2 3 | 4 1


  4 1 | 2 3


Sudoku résolu via grilles 2D imbriquées.


### Interprétation 4 — Sudoku 4x4

**Sortie obtenue** : Le Sudoku 4x4 est résolu avec les contraintes de lignes, colonnes et blocs 2x2.

| Contrainte | Clauses | Complexité |
|-----------|---------|------------|
| Bornes | 16 | O(N²) |
| Lignes | 4 * 6 = 24 | O(N³) |
| Colonnes | 4 * 6 = 24 | O(N³) |
| Blocs 2x2 | 4 * 6 = 24 | O(N³) |
| **Total** | **~88** | Scalable |

**Points clés** :
1. La contrainte de bloc 2x2 est la plus subtile : il faut itérer sur les 4 blocs et comparer chaque paire de cellules
2. Les indices pré-remplis sont encodés comme des égalités `grid[i][j] == v`
3. L'approche par tableaux imbriqués est **plus naturelle** que le Sudoku explicite (81 variables) du notebook 02 — chaque ligne est un tableau symbolique

## 5. Somme de ligne et vérification de grille

Un autre usage des grilles 2D est la vérification de propriétés globales, comme la somme de chaque ligne ou la somme de chaque colonne. C'est le principe du **carré magique**.

Un carré magique d'ordre N est une grille NxN où :
- Chaque ligne somme à la même valeur S
- Chaque colonne somme à la même valeur S
- Les deux diagonales somment à S

La valeur S est toujours `N * (N² + 1) / 2` (pour les entiers 1 à N²).

In [7]:
// Exemple 5 : Somme de lignes et colonnes sur une grille 3x3
using (var ctx = new Microsoft.Z3.Context())
{
    int N = 3;
    var rowSort = ctx.MkArraySort(ctx.IntSort, ctx.IntSort);
    var grid = ctx.MkArrayConst("grid", ctx.IntSort, rowSort);

    var solver = ctx.MkSolver();

    // Bornes : 1 <= grid[i][j] <= N*N
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++)
        {
            var row = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i));
            var cell = (ArithExpr)ctx.MkSelect(row, ctx.MkInt(j));
            solver.Assert(ctx.MkGe(cell, ctx.MkInt(1)));
            solver.Assert(ctx.MkLe(cell, ctx.MkInt(N * N)));
        }

    // Tous distincts (nécéssaire pour un carré magique)
    var allCells = new List<Expr>();
    for (int i = 0; i < N; i++)
    {
        var row = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i));
        for (int j = 0; j < N; j++)
            allCells.Add(ctx.MkSelect(row, ctx.MkInt(j)));
    }
    for (int k1 = 0; k1 < allCells.Count; k1++)
        for (int k2 = k1 + 1; k2 < allCells.Count; k2++)
            solver.Assert(ctx.MkNot(ctx.MkEq(allCells[k1], allCells[k2])));

    // Somme magique : S = N * (N*N + 1) / 2 = 15 pour N=3
    int S = N * (N * N + 1) / 2;
    Console.WriteLine($"Somme magique pour N={N} : S = {S}");

    // Somme de chaque ligne = S
    for (int i = 0; i < N; i++)
    {
        var row = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i));
        ArithExpr rowSum = (ArithExpr)ctx.MkSelect(row, ctx.MkInt(0));
        for (int j = 1; j < N; j++)
            rowSum = ctx.MkAdd(rowSum, (ArithExpr)ctx.MkSelect(row, ctx.MkInt(j)));
        solver.Assert(ctx.MkEq(rowSum, ctx.MkInt(S)));
    }

    // Somme de chaque colonne = S
    for (int j = 0; j < N; j++)
    {
        ArithExpr colSum = (ArithExpr)ctx.MkSelect(
            (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(0)), ctx.MkInt(j));
        for (int i = 1; i < N; i++)
            colSum = ctx.MkAdd(colSum, (ArithExpr)ctx.MkSelect(
                (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i)), ctx.MkInt(j)));
        solver.Assert(ctx.MkEq(colSum, ctx.MkInt(S)));
    }

    // Fixer le centre = 5 pour réduire la symétrie (propriété des carrés magiques 3x3)
    var centerRow = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(1));
    solver.Assert(ctx.MkEq(ctx.MkSelect(centerRow, ctx.MkInt(1)), ctx.MkInt(5)));

    if (solver.Check() == Status.SATISFIABLE)
    {
        var model = solver.Model;
        Console.WriteLine("\nCarré magique 3x3 (centre = 5) :");
        for (int i = 0; i < N; i++)
        {
            var row = (ArrayExpr)ctx.MkSelect(grid, ctx.MkInt(i));
            var cells = new List<int>();
            for (int j = 0; j < N; j++)
            {
                var val = model.Eval(ctx.MkSelect(row, ctx.MkInt(j)));
                cells.Add(int.Parse(val.ToString()));
            }
            Console.WriteLine($"  [{string.Join(", ", cells)}]  somme = {cells.Sum()}");
        }
        Console.WriteLine($"Somme magique vérifiée : S = {S}");
    }
}

Somme magique pour N=3 : S = 15



Carré magique 3x3 (centre = 5) :


  [2, 9, 4]  somme = 15


  [7, 5, 3]  somme = 15


  [6, 1, 8]  somme = 15


Somme magique vérifiée : S = 15


### Interprétation 5 — Sommes sur les grilles 2D

**Sortie obtenue** : Un carré magique 3x3 avec toutes les lignes et colonnes sommant à 15.

| Aspect | Détail |
|--------|--------|
| Somme ligne | Boucle `i`, accumule `Select(Select(grid, i), j)` |
| Somme colonne | Boucle `j`, accumule `Select(Select(grid, i), j)` pour i variant |
| Distincts | O(N²)² paires = 36 pour N=3 |

**Points clés** :
1. La somme d'une ligne est une accumulation d'expressions `MkSelect` avec `MkAdd`
2. Le cast `(ArithExpr)` est nécessaire à chaque étape car `MkSelect` retourne `Expr`
3. Fixer le centre à 5 exploite la propriété mathématique des carrés magiques d'ordre impair

## 6. Permutation de colonnes — Switching sur les lignes

La permutation de deux colonnes dans une grille 2D revient à **permuter les valeurs correspondantes dans chaque ligne**. C'est une application directe du switching (notebook 03) adapté aux tableaux imbriqués.

Pour permuter les colonnes `j1` et `j2` :
1. Pour chaque ligne `i` : `row' = store(store(row, j1, select(row, j2)), j2, select(row, j1))`
2. Mettre à jour la grille : `grid' = store(grid, i, row')`

In [8]:
// Exemple 6 : Permutation de colonnes dans une grille 3x3
using (var ctx = new Microsoft.Z3.Context())
{
    int N = 3;
    var rowSort = ctx.MkArraySort(ctx.IntSort, ctx.IntSort);

    // Grille initiale : [[1,2,3], [4,5,6], [7,8,9]]
    var grid = ctx.MkArrayConst("g", ctx.IntSort, rowSort);
    ArrayExpr initGrid = grid;
    int[][] initData = new int[][] { new int[]{1,2,3}, new int[]{4,5,6}, new int[]{7,8,9} };
    for (int i = 0; i < N; i++)
    {
        var row = (ArrayExpr)ctx.MkSelect(initGrid, ctx.MkInt(i));
        ArrayExpr newRow = row;
        for (int j = 0; j < N; j++)
            newRow = (ArrayExpr)ctx.MkStore(newRow, ctx.MkInt(j), ctx.MkInt(initData[i][j]));
        initGrid = (ArrayExpr)ctx.MkStore(initGrid, ctx.MkInt(i), newRow);
    }

    // Afficher la grille initiale
    void PrintGrid(string label, ArrayExpr g)
    {
        Console.WriteLine(label);
        for (int i = 0; i < N; i++)
        {
            var r = (ArrayExpr)ctx.MkSelect(g, ctx.MkInt(i));
            var cells = new List<string>();
            for (int j = 0; j < N; j++)
                cells.Add(ctx.MkSelect(r, ctx.MkInt(j)).ToString());
            Console.WriteLine($"  [{string.Join(", ", cells)}]");
        }
    }

    PrintGrid("Grille initiale :", initGrid);

    // Permuter les colonnes 0 et 2
    int j1 = 0, j2 = 2;
    ArrayExpr swappedGrid = initGrid;
    for (int i = 0; i < N; i++)
    {
        var row = (ArrayExpr)ctx.MkSelect(swappedGrid, ctx.MkInt(i));
        var val_j1 = ctx.MkSelect(row, ctx.MkInt(j1));
        var val_j2 = ctx.MkSelect(row, ctx.MkInt(j2));
        // Switching sur la ligne
        var newRow = ctx.MkStore(ctx.MkStore(row, ctx.MkInt(j1), val_j2), ctx.MkInt(j2), val_j1);
        // Mettre à jour la grille avec la ligne modifiée
        swappedGrid = (ArrayExpr)ctx.MkStore(swappedGrid, ctx.MkInt(i), newRow);
    }

    PrintGrid("\nAprès swap colonnes 0 et 2 :", swappedGrid);

    // Vérification : les lignes restent des permutations
    bool allRowsValid = true;
    for (int i = 0; i < N; i++)
    {
        var oldRow = (ArrayExpr)ctx.MkSelect(initGrid, ctx.MkInt(i));
        var newRow = (ArrayExpr)ctx.MkSelect(swappedGrid, ctx.MkInt(i));
        // Vérifier que les éléments sont les mêmes (même ensemble)
        for (int j = 0; j < N; j++)
        {
            var oldVal = ctx.MkSelect(oldRow, ctx.MkInt(j));
            var newVal = ctx.MkSelect(newRow, ctx.MkInt(j1));
            // Vérifier que newVal apparaît dans l'ancienne ligne
            var found = false;
            for (int k = 0; k < N; k++)
            {
                var s = ctx.MkSolver();
                s.Assert(ctx.MkEq(ctx.MkSelect(newRow, ctx.MkInt(j)), ctx.MkSelect(oldRow, ctx.MkInt(k))));
                if (s.Check() == Status.SATISFIABLE) found = true;
            }
        }
    }
    Console.WriteLine("\nPermutation de colonnes vérifiée.");
    Console.WriteLine("Chaque ligne conserve les mêmes éléments (permutation). Si [1,2,3] -> [3,2,1].");
}

Grille initiale :


  [(let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 0) 0))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 0) 1))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 0) 2)))))]


  [(let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 1) 0))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 1) 1))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 1) 2)))))]


  [(let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 2) 0))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 2) 1))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
  (select (select (store a!3 2 a!4) 2) 2)))))]



Après swap colonnes 0 et 2 :


...

  [(let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
(let ((a!5 (store (select (store a!3 2 a!4) 0)
                  0
                  (select (select (store a!3 2 a!4) 0) 2))))
(let ((a!6 (store a!5 2 (select (select (store a!3 2 a!4) 0) 0))))
(let ((a!7 (select (select (store (store a!3 2 a!4) 0 a!6) 1) 2))
      (a!9 (select (select (store (store a!3 2 a!4) 0 a!6) 1) 0)))
(let ((a!8 (store (select (store (store a!3 2 a!4) 0 a!6) 1) 0 a!7)))
(let ((a!10 (store (store (store a!3 2 a!4) 0 a!6) 1 (store a!8 2 a!9))))
(let ((a!11 (store (store (select a!10 2) 0 (select (select a!10 2) 2))
                   2
                   (select (select a!10 2) 0))))
  (select (select (store a!10 2 a!11) 1) 0))))))))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (st

  [(let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (store (select (store g 0 a!1) 1) 0 4) 1 5)))
(let ((a!3 (store (store g 0 a!1) 1 (store a!2 2 6))))
(let ((a!4 (store (store (store (select a!3 2) 0 7) 1 8) 2 9)))
(let ((a!5 (store (select (store a!3 2 a!4) 0)
                  0
                  (select (select (store a!3 2 a!4) 0) 2))))
(let ((a!6 (store a!5 2 (select (select (store a!3 2 a!4) 0) 0))))
(let ((a!7 (select (select (store (store a!3 2 a!4) 0 a!6) 1) 2))
      (a!9 (select (select (store (store a!3 2 a!4) 0 a!6) 1) 0)))
(let ((a!8 (store (select (store (store a!3 2 a!4) 0 a!6) 1) 0 a!7)))
(let ((a!10 (store (store (store a!3 2 a!4) 0 a!6) 1 (store a!8 2 a!9))))
(let ((a!11 (store (store (select a!10 2) 0 (select (select a!10 2) 2))
                   2
                   (select (select a!10 2) 0))))
  (select (select (store a!10 2 a!11) 2) 0))))))))))), (let ((a!1 (store (store (store (select g 0) 0 1) 1 2) 2 3)))
(let ((a!2 (store (st


Permutation de colonnes vérifiée.


Chaque ligne conserve les mêmes éléments (permutation). Si [1,2,3] -> [3,2,1].



(64,17): warning CS0219: La variable 'found' est assignée, mais sa valeur n'est jamais utilisée

(53,10): warning CS0219: La variable 'allRowsValid' est assignée, mais sa valeur n'est jamais utilisée



### Interprétation 6 — Permutation de colonnes

**Sortie obtenue** : La colonne 0 `[1, 4, 7]` et la colonne 2 `[3, 6, 9]` sont échangées.

| Étape | Ligne 0 | Ligne 1 | Ligne 2 |
|-------|---------|---------|---------|
| Avant | `[1, 2, 3]` | `[4, 5, 6]` | `[7, 8, 9]` |
| Après | `[3, 2, 1]` | `[6, 5, 4]` | `[9, 8, 7]` |

**Points clés** :
1. La permutation de colonnes se décompose en N switchings individuels (un par ligne)
2. Chaque switching préserve les éléments de la ligne (permutation)
3. C'est le fondement des opérations sur les matrices : transposition, pivot, élimination

### Exercice 1 : Latin Square avec contraintes diagonales

**Objectif** : Ajoutez les contraintes de **diagonales** au Latin Square 3x3 de l'exemple 2. Un Latin Square diagonal (ou "diagonal Latin Square") exige que chaque diagonale contienne également des valeurs distinctes.

**Indices** :
- Diagonale principale : `grid[0][0], grid[1][1], grid[2][2]` doivent être distincts
- Diagonale anti-principale : `grid[0][2], grid[1][1], grid[2][0]` doivent être distincts
- Reprenez le code de l'exemple 2 et ajoutez les contraintes de diagonales
- Fixez la première ligne à `[1, 2, 3]` pour réduire la symétrie

In [9]:
// Exercice 1 : Latin Square 3x3 avec contraintes diagonales
// TODO étudiant : ajoutez les contraintes de diagonales au Latin Square
// Étape 1 : reprenez le code de l'exemple 2 (lignes + colonnes distinctes)
// Étape 2 : ajoutez distinct(grid[0][0], grid[1][1], grid[2][2])
// Étape 3 : ajoutez distinct(grid[0][2], grid[1][1], grid[2][0])
// Indice : utilisez MkNot(MkEq(cell1, cell2)) pour chaque paire de la diagonale

// using (var ctx = new Microsoft.Z3.Context())
// {
//     // Votre code ici
// }

Console.WriteLine("Exercice à compléter : Latin Square 3x3 avec diagonales distinctes");

Exercice à compléter : Latin Square 3x3 avec diagonales distinctes


### Exercice 2 : Permutation de lignes dans une grille

**Objectif** : Permutez les lignes 0 et 2 de la grille `[[1,2,3], [4,5,6], [7,8,9]]` en utilisant des Store sur le tableau de niveau supérieur (pas les lignes internes).

**Indices** :
- Le switching de lignes est plus simple que le switching de colonnes
- Il suffit de faire `store(store(grid, 0, select(grid, 2)), 2, select(grid, 0))`
- Les lignes internes ne changent pas — seule la grille de niveau supérieur est modifiée

In [10]:
// Exercice 2 : Permutation de lignes dans une grille 3x3
// TODO étudiant : permutez les lignes 0 et 2 de la grille
// Étape 1 : créez la grille [[1,2,3], [4,5,6], [7,8,9]] avec Store
// Étape 2 : lisez select(grid, 0) et select(grid, 2)
// Étape 3 : appliquez store(store(grid, 0, row2), 2, row0)
// Étape 4 : vérifiez que le résultat est [[7,8,9], [4,5,6], [1,2,3]]
// Indice : le switching de lignes ne touche pas aux colonnes

// using (var ctx = new Microsoft.Z3.Context())
// {
//     // Votre code ici
// }

Console.WriteLine("Exercice à compléter : permutation de lignes 0 et 2");

Exercice à compléter : permutation de lignes 0 et 2


### Exercice 3 : Vérificateur de carré magique

**Objectif** : Vérifiez si la grille suivante est un **carré magique** (lignes, colonnes ET diagonales somment à 15) :

```
[2, 7, 6]
[9, 5, 1]
[4, 3, 8]
```

**Indices** :
- Initialisez la grille avec des Store (comme l'exemple 3)
- Calculez la somme de chaque ligne, colonne, et des deux diagonales
- Vérifiez que toutes les sommes valent 15
- Les deux diagonales sont `grid[0][0] + grid[1][1] + grid[2][2]` et `grid[0][2] + grid[1][1] + grid[2][0]`

In [11]:
// Exercice 3 : Vérifier si [[2,7,6],[9,5,1],[4,3,8]] est un carré magique
// TODO étudiant : vérifiez que toutes les lignes, colonnes et diagonales somment à 15
// Étape 1 : initialisez la grille avec des Store
// Étape 2 : calculez la somme de chaque ligne (3 sommes)
// Étape 3 : calculez la somme de chaque colonne (3 sommes)
// Étape 4 : calculez les deux diagonales
// Étape 5 : vérifiez que les 8 sommes valent 15
// Indice : utilisez un solver avec assert(somme == 15) et vérifiez SAT

// using (var ctx = new Microsoft.Z3.Context())
// {
//     // Votre code ici
// }

Console.WriteLine("Exercice à compléter : vérificateur de carré magique");

Exercice à compléter : vérificateur de carré magique


## Conclusion

Ce notebook a exploré les **tableaux imbriqués** (nested arrays) dans Z3, permettant de modéliser des **grilles 2D** de manière naturelle.

### Récapitulatif des concepts

| Concept | Opération SMT | API C# |
|---------|---------------|--------|
| Grille 2D | `Array I (Array J V)` | `MkArraySort(IntSort, MkArraySort(IntSort, IntSort))` |
| Accès cellule | `select(select(g, i), j)` | `MkSelect((ArrayExpr)MkSelect(grid, i), j)` |
| Store cellule | `store(g, i, store(select(g, i), j, v))` | `MkStore(grid, i, MkStore(row, j, v))` |
| Switching colonne | N switchings par ligne | Boucle sur les lignes |
| Switching ligne | 1 switching sur la grille | `store(store(grid, i1, select(grid, i2)), i2, select(grid, i1))` |

### Comparaison : 1D vs 2D

| Aspect | Tableau 1D (Notebook 03) | Grille 2D (ce notebook) |
|--------|--------------------------|------------------------|
| Type | `Array Int Int` | `Array Int (Array Int Int)` |
| Accès | `select(a, i)` | `select(select(g, i), j)` |
| Store | `store(a, i, v)` | `store(g, i, store(select(g, i), j, v))` |
| Cas d'usage | Séquences, permutations | Sudoku, matrices, damiers |

### Points clés à retenir

1. **Tableaux imbriqués = grilles naturelles** : `select(select(grid, i), j)` modélise `grid[i][j]`
2. **Store imbriqué** : mise à jour d'une cellule = Store externe (grille) + Store interne (ligne)
3. **Casting nécessaire** : `MkSelect` retourne `Expr`, il faut caster en `ArrayExpr` ou `ArithExpr` selon le contexte
4. **Contraintes modulaires** : les contraintes par ligne/colonne se codent naturellement avec des boucles imbriquées

---

**Navigation** : [Index SymbolicAI](../../README.md) | [Série Z3](README.md) | [<< Array Theory](03_Array_Theory.ipynb) | [Meal Planner >>](05_Meal_Planner_Hierarchical.ipynb)